# 6.1 TensorBoard 可视化基础

本 notebook 介绍 PyTorch 中使用 TensorBoard 进行训练过程可视化的方法。

**学习目标：**
- 掌握 SummaryWriter 的创建和配置
- 学会使用 add_scalar / add_scalars 记录标量数据
- 学会使用 add_histogram 记录参数分布
- 学会使用 add_image / add_images 记录图像
- 学会使用 add_graph 可视化模型结构
- 掌握完整训练循环中的 TensorBoard 日志记录

## 1. 安装与导入

TensorBoard 是 TensorFlow 的可视化工具，PyTorch 通过 `torch.utils.tensorboard` 提供了原生支持。

In [ ]:
# 安装 tensorboard（如果尚未安装）
# !pip install tensorboard

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.tensorboard import SummaryWriter
import os
import shutil

print(f"PyTorch 版本: {torch.__version__}")

## 2. SummaryWriter 创建

SummaryWriter 是 TensorBoard 的核心类，负责将数据写入日志文件。

**关键参数：**
- `log_dir`: 日志目录路径
- `comment`: 在默认目录名后附加的注释
- `flush_secs`: 自动刷新间隔（秒）

In [ ]:
# 清理之前的日志
log_base = "./tb_logs"
if os.path.exists(log_base):
    shutil.rmtree(log_base)

# 方式1: 指定 log_dir
writer1 = SummaryWriter(log_dir=os.path.join(log_base, "experiment_1"))
print(f"日志目录: {writer1.log_dir}")

# 方式2: 使用 comment 后缀（自动生成带时间戳的目录）
writer2 = SummaryWriter(comment="_lr0.01_bs32")
print(f"带注释的日志目录: {writer2.log_dir}")

# 方式3: 指定 flush_secs（控制写入频率）
writer3 = SummaryWriter(
    log_dir=os.path.join(log_base, "experiment_3"),
    flush_secs=30  # 每30秒自动刷新
)
print(f"自定义刷新: {writer3.log_dir}")

# 关闭多余的 writer
writer2.close()
writer3.close()

## 3. add_scalar：记录单个标量值

`add_scalar(tag, scalar_value, global_step)` 用于记录单条曲线。

**tag 分组技巧：** 使用 `/` 分隔，如 `Loss/train`、`Loss/val`，TensorBoard 会自动分组显示。

In [ ]:
writer = SummaryWriter(log_dir=os.path.join(log_base, "scalar_demo"))

# 模拟训练和验证损失
for epoch in range(100):
    # 模拟训练损失（指数衰减 + 噪声）
    train_loss = 2.0 * np.exp(-0.03 * epoch) + np.random.normal(0, 0.05)
    # 模拟验证损失（衰减较慢）
    val_loss = 2.0 * np.exp(-0.02 * epoch) + np.random.normal(0, 0.08)
    # 模拟准确率
    train_acc = 1.0 - np.exp(-0.03 * epoch) + np.random.normal(0, 0.02)
    
    # 使用 tag 分组: Loss/train 和 Loss/val 会在同一组
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/val", val_loss, epoch)
    writer.add_scalar("Accuracy/train", train_acc, epoch)

writer.close()
print("标量数据已写入")
print("提示: 使用 'Loss/train' 和 'Loss/val' 的分组命名，TensorBoard 会自动归类")

## 4. add_scalars：在同一图表中绘制多条曲线

`add_scalars(main_tag, tag_scalar_dict, global_step)` 将多条曲线绘制在同一个图表中。

与 `add_scalar` 分组的区别：`add_scalars` 直接在同一张图上叠加，`add_scalar` 分组则在同一面板中并排显示。

In [ ]:
writer = SummaryWriter(log_dir=os.path.join(log_base, "scalars_demo"))

# 模拟不同学习率的训练损失
for step in range(100):
    writer.add_scalars(
        main_tag="Loss/compare",
        tag_scalar_dict={
            "lr=0.1": 2.0 * np.exp(-0.05 * step) + np.random.normal(0, 0.03),
            "lr=0.01": 2.0 * np.exp(-0.03 * step) + np.random.normal(0, 0.03),
            "lr=0.001": 2.0 * np.exp(-0.01 * step) + np.random.normal(0, 0.03),
        },
        global_step=step
    )

writer.close()
print("多条曲线数据已写入")
print("三条学习率曲线将在同一张图表中显示")

## 5. add_histogram：参数分布跟踪

`add_histogram(tag, values, global_step)` 用于记录张量的分布直方图。

常见用途：跟踪权重、梯度、激活值的分布变化。

In [ ]:
writer = SummaryWriter(log_dir=os.path.join(log_base, "histogram_demo"))

# 模拟权重分布随训练变化
for epoch in range(50):
    # 模拟权重: 从均匀分布逐渐变为正态分布
    alpha = epoch / 50.0
    uniform_part = torch.rand(1000) * 2 - 1  # [-1, 1] 均匀分布
    normal_part = torch.randn(1000) * 0.5     # 正态分布
    weights = (1 - alpha) * uniform_part + alpha * normal_part
    
    # 模拟梯度: 随训练逐渐变小
    gradients = torch.randn(1000) * (1.0 - 0.8 * alpha)
    
    writer.add_histogram("weights/layer1", weights, epoch)
    writer.add_histogram("gradients/layer1", gradients, epoch)

writer.close()
print("直方图数据已写入")
print("可以在 TensorBoard 的 HISTOGRAMS 面板中查看分布变化")

## 6. add_image / add_images：图像可视化

- `add_image(tag, img_tensor, global_step)`: 记录单张图像，形状为 (C, H, W)
- `add_images(tag, img_tensor, global_step)`: 记录批量图像，形状为 (N, C, H, W)

In [ ]:
writer = SummaryWriter(log_dir=os.path.join(log_base, "image_demo"))

# 单张图像: (C, H, W) 格式
single_image = torch.rand(3, 64, 64)  # 随机 RGB 图像
writer.add_image("single_image/random", single_image, global_step=0)

# 创建一张简单的渐变图像
gradient_image = torch.zeros(3, 64, 64)
for i in range(64):
    gradient_image[0, :, i] = i / 64.0  # 红色渐变
    gradient_image[1, i, :] = i / 64.0  # 绿色渐变
writer.add_image("single_image/gradient", gradient_image, global_step=0)

# 批量图像: (N, C, H, W) 格式
batch_images = torch.rand(16, 3, 32, 32)  # 16 张随机图像
writer.add_images("batch_images/random", batch_images, global_step=0)

writer.close()
print("图像数据已写入")
print("add_image: 接受 (C, H, W) 格式")
print("add_images: 接受 (N, C, H, W) 格式")

## 7. add_graph：模型结构可视化

`add_graph(model, input_to_model)` 将模型的计算图写入 TensorBoard，可在 GRAPHS 面板中查看网络结构。

In [ ]:
# 定义一个简单的 CNN 模型
class SimpleCNN(nn.Module):
    """用于演示 add_graph 的简单 CNN。"""
    
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 32 -> 16
        x = self.pool(F.relu(self.conv2(x)))  # 16 -> 8
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN()
writer = SummaryWriter(log_dir=os.path.join(log_base, "graph_demo"))

# 需要提供一个样本输入
dummy_input = torch.randn(1, 3, 32, 32)
writer.add_graph(model, dummy_input)

writer.close()
print("模型计算图已写入")
print("可在 TensorBoard 的 GRAPHS 面板中查看网络结构")

## 8. add_figure：嵌入 Matplotlib 图表

`add_figure(tag, figure, global_step)` 将 matplotlib 图表作为图像嵌入 TensorBoard。

In [ ]:
import matplotlib.pyplot as plt

writer = SummaryWriter(log_dir=os.path.join(log_base, "figure_demo"))

# 创建 matplotlib 图表
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 子图1: 损失曲线
epochs = np.arange(100)
train_loss = 2.0 * np.exp(-0.03 * epochs)
val_loss = 2.0 * np.exp(-0.02 * epochs) + 0.3
axes[0].plot(epochs, train_loss, label="Train")
axes[0].plot(epochs, val_loss, label="Val")
axes[0].set_title("Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

# 子图2: 参数分布
weights = np.random.randn(1000)
axes[1].hist(weights, bins=50, alpha=0.7)
axes[1].set_title("Weight Distribution")
axes[1].set_xlabel("Value")
axes[1].set_ylabel("Count")

plt.tight_layout()

# 写入 TensorBoard
writer.add_figure("analysis/training_overview", fig, global_step=0)

writer.close()
plt.close(fig)
print("Matplotlib 图表已嵌入 TensorBoard")

## 9. 其他常用方法

TensorBoard 还支持多种数据类型的记录：

In [ ]:
writer = SummaryWriter(log_dir=os.path.join(log_base, "other_demo"))

# ---- add_text: 记录文本信息 ----
writer.add_text(
    "experiment/config",
    "lr=0.01, batch_size=32, optimizer=Adam",
    global_step=0
)
writer.add_text(
    "experiment/notes",
    "第一次实验：使用默认超参数",
    global_step=0
)
print("add_text: 记录文本信息，适合记录实验配置和备注")

# ---- add_embedding: 高维数据降维可视化 ----
# 生成一些带标签的高维数据
features = torch.randn(100, 128)  # 100 个样本，128 维特征
labels = torch.randint(0, 5, (100,))  # 5 类标签
label_names = [f"class_{labels[i].item()}" for i in range(100)]

writer.add_embedding(
    features,
    metadata=label_names,
    tag="feature_embedding",
    global_step=0
)
print("add_embedding: 高维特征降维可视化（PCA/t-SNE/UMAP）")

# ---- add_pr_curve: PR 曲线 ----
labels_binary = torch.randint(0, 2, (100,))  # 二分类标签
predictions = torch.rand(100)  # 预测概率
writer.add_pr_curve(
    "pr_curve/binary",
    labels_binary,
    predictions,
    global_step=0
)
print("add_pr_curve: 精确率-召回率曲线")

# ---- add_hparams: 超参数对比 ----
writer.add_hparams(
    hparam_dict={"lr": 0.01, "batch_size": 32, "optimizer": "Adam"},
    metric_dict={"hparam/accuracy": 0.85, "hparam/loss": 0.35}
)
print("add_hparams: 记录超参数和对应的指标，方便超参数对比")

writer.close()

## 10. 完整训练循环示例

将 TensorBoard 集成到一个完整的训练循环中：

In [ ]:
# 定义模型
class SimpleNet(nn.Module):
    """用于演示完整训练循环的简单网络。"""
    
    def __init__(self, input_dim: int = 10, hidden_dim: int = 32, output_dim: int = 2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 生成模拟数据
torch.manual_seed(42)
n_samples = 500
x_data = torch.randn(n_samples, 10)
y_data = (x_data[:, 0] + x_data[:, 1] > 0).long()  # 简单的二分类

# 划分训练集和验证集
train_x, val_x = x_data[:400], x_data[400:]
train_y, val_y = y_data[:400], y_data[400:]

print(f"训练集: {train_x.shape}, 验证集: {val_x.shape}")
print(f"正样本比例: {y_data.float().mean():.2f}")

In [ ]:
# 训练循环 + TensorBoard 记录
model = SimpleNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

writer = SummaryWriter(log_dir=os.path.join(log_base, "full_training"))

# 记录模型结构
writer.add_graph(model, torch.randn(1, 10))

# 记录初始超参数
writer.add_text("config", "lr=0.01, hidden=32, dropout=0.3, optimizer=Adam")

n_epochs = 50
for epoch in range(n_epochs):
    # --- 训练阶段 ---
    model.train()
    optimizer.zero_grad()
    
    train_output = model(train_x)
    train_loss = criterion(train_output, train_y)
    train_loss.backward()
    optimizer.step()
    
    train_pred = train_output.argmax(dim=1)
    train_acc = (train_pred == train_y).float().mean()
    
    # --- 验证阶段 ---
    model.eval()
    with torch.no_grad():
        val_output = model(val_x)
        val_loss = criterion(val_output, val_y)
        val_pred = val_output.argmax(dim=1)
        val_acc = (val_pred == val_y).float().mean()
    
    # --- 记录到 TensorBoard ---
    # 损失
    writer.add_scalar("Loss/train", train_loss.item(), epoch)
    writer.add_scalar("Loss/val", val_loss.item(), epoch)
    
    # 准确率
    writer.add_scalar("Accuracy/train", train_acc.item(), epoch)
    writer.add_scalar("Accuracy/val", val_acc.item(), epoch)
    
    # 学习率
    writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)
    
    # 每 10 个 epoch 记录参数分布
    if epoch % 10 == 0:
        for name, param in model.named_parameters():
            writer.add_histogram(f"params/{name}", param.data, epoch)
            if param.grad is not None:
                writer.add_histogram(f"grads/{name}", param.grad, epoch)
    
    # 打印进度
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

# 记录最终超参数和指标
writer.add_hparams(
    hparam_dict={"lr": 0.01, "hidden_dim": 32, "dropout": 0.3},
    metric_dict={"hparam/final_train_acc": train_acc.item(),
                 "hparam/final_val_acc": val_acc.item()}
)

writer.close()
print("\n训练完成，所有数据已写入 TensorBoard")

## 11. 启动 TensorBoard

在命令行中运行以下命令查看日志：

```bash
# 启动 TensorBoard
tensorboard --logdir=./tb_logs

# 指定端口
tensorboard --logdir=./tb_logs --port=6007

# 在 Jupyter 中内嵌显示
%load_ext tensorboard
%tensorboard --logdir=./tb_logs
```

**TensorBoard 面板说明：**

| 面板 | 对应方法 | 用途 |
|------|----------|------|
| SCALARS | add_scalar / add_scalars | 标量曲线 |
| IMAGES | add_image / add_images | 图像展示 |
| HISTOGRAMS | add_histogram | 分布直方图 |
| GRAPHS | add_graph | 模型结构 |
| TEXT | add_text | 文本记录 |
| PROJECTOR | add_embedding | 高维数据降维 |
| PR CURVES | add_pr_curve | PR 曲线 |
| HPARAMS | add_hparams | 超参数对比 |

In [ ]:
# 清理演示产生的日志文件
if os.path.exists(log_base):
    shutil.rmtree(log_base)
    print(f"已清理日志目录: {log_base}")

# 清理 runs 目录（writer2 使用 comment 参数自动创建的）
import glob
for d in glob.glob("runs/*_lr0.01_bs32"):
    shutil.rmtree(d)
    print(f"已清理: {d}")

## 12. 总结

### SummaryWriter 核心方法

| 方法 | 参数 | 用途 |
|------|------|------|
| `add_scalar` | tag, value, step | 记录单条标量曲线 |
| `add_scalars` | main_tag, dict, step | 多条曲线叠加在同一图表 |
| `add_histogram` | tag, values, step | 记录分布直方图 |
| `add_image` | tag, img(C,H,W), step | 记录单张图像 |
| `add_images` | tag, imgs(N,C,H,W), step | 记录批量图像 |
| `add_graph` | model, input | 可视化模型结构 |
| `add_figure` | tag, figure, step | 嵌入 matplotlib 图表 |
| `add_text` | tag, text, step | 记录文本 |
| `add_embedding` | features, metadata | 高维数据降维可视化 |
| `add_hparams` | hparam_dict, metric_dict | 超参数对比 |

### 最佳实践
1. 使用 `/` 分组 tag（如 `Loss/train`、`Loss/val`）
2. 在训练循环中定期记录参数分布（`add_histogram`）
3. 记录模型结构（`add_graph`）和实验配置（`add_text` / `add_hparams`）
4. 使用完毕后调用 `writer.close()` 确保数据写入
5. 不同实验使用不同的 `log_dir`，方便对比

---

## 练习题

**练习1（基础）：** 模拟一个 200 个 epoch 的训练过程，记录以下信息到 TensorBoard：
- 训练损失和验证损失（使用 tag 分组）
- 训练准确率和验证准确率
- 模拟一个学习率衰减过程（每 50 个 epoch 减半）

**练习2（进阶）：** 使用 `add_scalars` 对比三种不同学习率（0.1、0.01、0.001）的训练损失曲线，分析哪种学习率收敛最好。

**练习3（挑战）：** 构建一个完整的训练管道：
- 定义一个多层网络
- 每个 epoch 记录损失、准确率、学习率
- 每 5 个 epoch 记录所有层的权重分布和梯度分布
- 记录模型结构和超参数